# Exercise 1
Build a Polite News Crawler: Implement a domain-specific scraper that collects
news articles about Artificial Intelligence. As the base of your crawler use BeautifulSoup.

**Tasks**  
• Crawl at least 3 different news websites.  
• Collect at least 200 articles and 100,000 words total.  
• Store URL, title, publication date, article text, domain, and crawl timestamp.  
• Respect robots.txt.  
• Implement rate limiting (≥ 1 second delay).  
• Avoid duplicate URLs.  


## Imports

In [75]:
import time
import json
import requests
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin
from urllib.robotparser import RobotFileParser
from datetime import datetime, UTC

## Check robots.txt
**AI Use:** Microsoft Copilot GPT-5.1  
*Prompt:* write me a short function to check the robot txt and use the following for it: rp = RobotFileParser()    

In [62]:
def robots_allowed(url, user_agent="*"):
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"

    rp = RobotFileParser()
    try:
        rp.set_url(robots_url)
        rp.read()
    except:
        return False  # safer default

    return rp.can_fetch(user_agent, url)

## fetch HTML with rate limiting
**AI Use:** Microsoft Copilot GPT-5.1  
*Prompt:* write me a short function to fetch htmls and use the following for it: r = requests.get(url, headers, timeout=10)

In [63]:
HEADERS = {"User-Agent": "Microsoft Edge/145.0.3800.82"}
RATE_LIMIT = 1.0  # seconds

def fetch_html(url):
    time.sleep(RATE_LIMIT)
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        r.raise_for_status()
        return r.text
    except:
        return None

## JSON

In [33]:
def write_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

## Definition of Keywords

In [34]:
AI_Keywords = ["artificial intelligence", "machine learning", "deep learning", "neural network", "ai", "ml", "dl", "nn", "gpt", "transformer", "llm", "large language model", "chatgpt", "bard", "gemini", "claude"]

In [35]:
def ai_related(text):
    text = text.lower()
    return any(keyword in text for keyword in AI_Keywords)

## Extraction from sides

### Website Ars Technica

#### Get links
**AI Use:** Microsoft Copilot GPT-5.1  
*Prompt:* write me a short function to get the link from a following website

In [64]:
def get_urls_from_sitemap(sitemap_url, must_contain=None, limit=None):
    """
    Parse a sitemap XML and return article URLs.
    Optionally filter by substrings in `must_contain`.
    """
    html = fetch_html(sitemap_url)
    if not html:
        return []

    urls = []
    try:
        root = ET.fromstring(html)
    except:
        return []

    ns = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"}

    for loc in root.iter():
        if loc.tag.endswith("loc"):
            url = loc.text.strip()
            if must_contain:
                if not any(sub in url for sub in must_contain):
                    continue
            urls.append(url)
            if limit and len(urls) >= limit:
                break

    return urls

In [65]:
def get_urls_from_rss(feed_url, limit=None):
    html = fetch_html(feed_url)
    if not html:
        return []

    urls = []
    root = ET.fromstring(html)

    for item in root.findall(".//item"):
        link = item.find("link")
        if link is not None and link.text:
            urls.append(link.text.strip())
            if limit and len(urls) >= limit:
                break

    return urls

In [54]:
def extract_links_arstechnica(html, base):
    soup = BeautifulSoup(html, "html.parser")
    links = []

    # Ars Technica uses multiple link types
    for a in soup.select("a.card, a.overlay-link, a.tease"):
        href = a.get("href")
        if href and href.startswith("/"):
            links.append(urljoin(base, href))

    return links

#### Get articles

In [66]:
def extract_article_arstechnica(html, url):
    soup = BeautifulSoup(html, "lxml")

    article = soup.select_one("article")
    if not article:
        return None

    title_el = article.select_one("h1")
    date_el = article.select_one("header time[datetime]")
    body_paras = article.select("div.post-content p")

    if not title_el or not body_paras:
        return None

    text = " ".join(p.get_text(" ", strip=True) for p in body_paras)

    return {
        "url": url,
        "title": title_el.get_text(strip=True),
        "date": date_el["datetime"] if date_el and date_el.has_attr("datetime") else None,
        "text": text,
        "domain": urlparse(url).netloc,
        "crawl_timestamp": datetime.now(UTC).isoformat()
    }

In [67]:
def extract_article_techcrunch(html, url):
    soup = BeautifulSoup(html, "lxml")

    article = soup.select_one("article")
    if not article:
        return None

    title_el = article.select_one("h1.article__title, h1")
    date_el = article.select_one("time")
    body_paras = article.select("div.article-content p")

    if not title_el or not body_paras:
        return None

    text = " ".join(p.get_text(" ", strip=True) for p in body_paras)

    return {
        "url": url,
        "title": title_el.get_text(strip=True),
        "date": date_el.get("datetime") if date_el else None,
        "text": text,
        "domain": urlparse(url).netloc,
        "crawl_timestamp": datetime.now(UTC).isoformat()
    }

In [68]:
def extract_article_reuters(html, url):
    soup = BeautifulSoup(html, "lxml")

    article = soup.select_one("article")
    if not article:
        return None

    title_el = article.select_one("h1")
    date_el = article.select_one("time")
    body_paras = article.select("p")

    if not title_el or not body_paras:
        return None

    text = " ".join(p.get_text(" ", strip=True) for p in body_paras)

    return {
        "url": url,
        "title": title_el.get_text(strip=True),
        "date": date_el.get("datetime") if date_el else None,
        "text": text,
        "domain": urlparse(url).netloc,
        "crawl_timestamp": datetime.now(UTC).isoformat()
    }

In [69]:
def crawl_urls(urls, article_extractor, output_path, max_articles=None):
    visited = set()
    collected = 0

    for url in urls:
        if max_articles and collected >= max_articles:
            break

        if url in visited:
            continue
        visited.add(url)

        if not robots_allowed(url):
            continue

        html = fetch_html(url)
        if not html:
            continue

        article = article_extractor(html, url)
        if not article:
            continue

        if not ai_related(article["title"] + " " + article["text"]):
            continue

        write_jsonl(output_path, article)

        collected += 1
        print(f"[{collected}] {article['domain']} — {article['title'][:80]}")

    print(f"Finished: {collected} articles collected from {article['domain'] if collected else 'this source'}.")

### Get list of URLs for each site

In [70]:
ars_urls = get_urls_from_sitemap(
    "https://arstechnica.com/sitemap.xml",
    must_contain=["/science/", "/information-technology/", "artificial-intelligence"],
    limit=400
)
len(ars_urls), ars_urls[:5]

(0, [])

In [71]:
tc_urls = get_urls_from_rss(
    "https://techcrunch.com/tag/artificial-intelligence/feed/",
    limit=400
)
len(tc_urls), tc_urls[:5]

NameError: name 'ET' is not defined

In [72]:
reuters_urls = get_urls_from_sitemap(
    "https://www.reuters.com/sitemap_technology.xml",
    must_contain=["artificial-intelligence", "/technology/"],
    limit=400
)
len(reuters_urls), reuters_urls[:5]


(0, [])

In [74]:
OUTPUT = "ai_news.jsonl"

crawl_urls(ars_urls,     extract_article_arstechnica, OUTPUT, max_articles=80)
#crawl_urls(tc_urls,      extract_article_techcrunch,  OUTPUT, max_articles=80)
crawl_urls(reuters_urls, extract_article_reuters,     OUTPUT, max_articles=80)

Finished: 0 articles collected from this source.
Finished: 0 articles collected from this source.
